# EEG Dementia Classification — Google Colab Pipeline

Runs feature extraction and/or model training for this repo in the cloud (no local PC needed).

**Before you start**
1. Runtime → Change runtime type → **CPU** is enough (GPU not required).
2. Pick a mode in the config cell below:
   - `full` — extract features from raw EEG, then train XGBoost + MLP
   - `train_only` — download precomputed features from Zenodo, then train
   - `smoke_test` — process 2 subjects per dataset (quick sanity check)
3. For `full` / `smoke_test`, upload raw EEG to Google Drive (see data cell).

Dataset: [Miltiadous et al. 2023](https://doi.org/10.3390/data8060095)

In [ ]:
# @title Configuration
MODE = "full"  # "full" | "train_only" | "smoke_test"

REPO_URL = "https://github.com/RandomPerson5571/ad_eeg.git"
REPO_BRANCH = "main"
PROJECT_DIR = "/content/ad_eeg"

# Google Drive folder that contains EEG_data/ (only needed for full / smoke_test)
# Example: /content/drive/MyDrive/EEG_Project/EEG_data
DRIVE_EEG_DATA_DIR = "/content/drive/MyDrive/EEG_Project/EEG_data"

# Where to copy results on Drive when the run finishes
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/EEG_Project/colab_outputs"

# Zenodo record ID for precomputed features (train_only mode)
ZENODO_RECORD_ID = None  # e.g. 1234567

TRAIN_MODELS = "xgboost,mlp"  # comma-separated: xgboost,mlp

In [ ]:
# @title Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# @title Clone repo and install dependencies
import os
import subprocess
import sys


def run(cmd, cwd=None):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, cwd=cwd)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {cmd}")


if os.path.exists(PROJECT_DIR):
    run(f"git -C {PROJECT_DIR} pull")
else:
    run(f"git clone --branch {REPO_BRANCH} {REPO_URL} {PROJECT_DIR}")

run(f"{sys.executable} -m pip install -q -r requirements.txt", cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# @title Link raw EEG from Drive (full / smoke_test only)
import os
from pathlib import Path

eeg_link = Path(PROJECT_DIR) / "EEG_data"

if MODE in ("full", "smoke_test"):
    drive_eeg = Path(DRIVE_EEG_DATA_DIR)
    if not drive_eeg.exists():
        raise FileNotFoundError(
            f"EEG data not found at {drive_eeg}.\n"
            "Upload the Miltiadous dataset so Drive contains:\n"
            "  EEG_data/dataset2/participants.tsv\n"
            "  EEG_data/dataset2/sub-001/eeg/...\n"
            "  EEG_data/dataset3/..."
        )

    if eeg_link.is_symlink() or eeg_link.exists():
        if eeg_link.is_symlink():
            eeg_link.unlink()
        elif eeg_link.is_dir():
            pass  # already a real folder in Colab
        else:
            eeg_link.unlink()

    if not eeg_link.exists():
        os.symlink(drive_eeg, eeg_link)

    !python scripts/download_data.py
else:
    print(f"Skipping EEG link (MODE={MODE})")

In [ ]:
# @title Run pipeline
import json
import os
import subprocess
import sys
from pathlib import Path

project = Path(PROJECT_DIR)
os.chdir(project)


def run_py(args):
    cmd = [sys.executable] + args
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


if MODE == "train_only":
    if not ZENODO_RECORD_ID:
        raise ValueError("Set ZENODO_RECORD_ID in the config cell for train_only mode.")
    run_py(["scripts/fetch_artifacts.py", "--record-id", str(ZENODO_RECORD_ID)])
    run_py(["scripts/run_pipeline.py", "--train", TRAIN_MODELS])

elif MODE == "smoke_test":
    run_py([
        "scripts/ingest_features.py",
        "--all-datasets",
        "--all",
        "--limit", "2",
    ])
    run_py(["scripts/run_pipeline.py", "--train", TRAIN_MODELS])

elif MODE == "full":
    run_py([
        "scripts/ingest_features.py",
        "--all-datasets",
        "--all",
    ])
    run_py(["scripts/run_pipeline.py", "--train", TRAIN_MODELS])

else:
    raise ValueError(f"Unknown MODE: {MODE}")

metrics_path = project / "results" / "metrics.json"
if metrics_path.exists():
    print("\n=== metrics.json ===")
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))
else:
    print("No metrics.json found — check logs above.")

In [ ]:
# @title Save outputs to Google Drive
import shutil
from datetime import datetime
from pathlib import Path

project = Path(PROJECT_DIR)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_root = Path(DRIVE_OUTPUT_DIR) / f"run_{stamp}_{MODE}"
out_root.mkdir(parents=True, exist_ok=True)

artifacts = [
    "parquet_files/all_features.parquet",
    "results/metrics.json",
    "results/metrics_xgboost.json",
    "results/metrics_mlp.json",
    "results/ingest_log.json",
    "results/preprocessing_config.json",
    "results/subject_splits.json",
    "classifier_models/saved_models/xgboost_eeg_classifier.joblib",
    "classifier_models/saved_models/eeg_mlp_classifier.joblib",
]

copied = []
for rel in artifacts:
    src = project / rel
    if not src.exists():
        continue
    dest = out_root / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dest)
    copied.append(rel)

print(f"Saved {len(copied)} file(s) to:\n  {out_root}")
for rel in copied:
    print(f"  - {rel}")

## Google Drive layout (for `full` mode)

Upload the downloaded Miltiadous dataset so your Drive looks like:

```
MyDrive/
  EEG_Project/
    EEG_data/
      dataset2/
        participants.tsv
        sub-001/eeg/sub-001_task-eyesclosed_eeg.set
        ...
      dataset3/
        participants.tsv
        sub-001/eeg/sub-001_task-photomark_eeg.set
        ...
    colab_outputs/          # created automatically after each run
```

Then set `DRIVE_EEG_DATA_DIR = "/content/drive/MyDrive/EEG_Project/EEG_data"` in the config cell.